<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #0f172a 0%, #1e3a8a 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.3);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🧊 TRELLIS.2 - Image to 3D Generator</h1>
  <h3 style='color: #93c5fd; margin: 0 0 5px 0; font-weight: 400;'>Kaggle GPU Edition - Created by <strong>AIQUEST Academy</strong></h3>
  <p style='color: #bfdbfe; margin: 0; text-align: center;'>Microsoft 4B-Parameter 3D Generation | PBR Textures | GLB Export</p>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-T4%2FP100-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />
  <br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
</div>

---

### What is this notebook?

**Image-to-3D Generation** using Microsoft's **TRELLIS.2** (4B parameters) — converts a single image into a fully textured 3D mesh with PBR materials (GLB export).

- **Model:** [Microsoft TRELLIS.2-4B](https://github.com/microsoft/TRELLIS.2) (MIT license)
- **Output:** GLB with PBR textures (Base Color, Metallic, Roughness, Alpha)
- **Installation:** Pre-compiled CUDA wheels — no compilation needed
- **Memory:** xformers attention backend + `expandable_segments` for T4 16GB

### Quick Start
1. **Settings → Accelerator → GPU T4 x2** (or P100)
2. **Turn on Internet** in the Settings sidebar
3. Run all cells in order
4. Open the public Gradio link from the final cell

**No HuggingFace token required** — DINOv3 is downloaded from a public mirror.

---
## Step 1: Environment Setup

In [ ]:
import os, gc, psutil

print('=== Kaggle Environment Setup ===')
print(f'RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB total, {psutil.virtual_memory().available / 1024**3:.1f} GB available')

os.system('echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1')
os.system('echo 1 | sudo tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1')
gc.collect()

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,garbage_collection_threshold:0.6'
os.environ['MALLOC_TRIM_THRESHOLD_'] = '0'
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPARSE_ATTN_BACKEND'] = 'xformers'

print('✅ Environment optimized!')

---
## Step 2: Clone TRELLIS.2 + Install Dependencies

Uses pre-compiled CUDA wheels (Torch 2.9.1 + Python 3.12 + CUDA 12.4) — no compilation needed.

In [ ]:
import subprocess
try:
    subprocess.run(['nvidia-smi'], check=True)
    print('✅ GPU Active!')
except Exception:
    print('WARNING: No GPU. Go to Settings -> Accelerator -> GPU T4 x2')

%cd /kaggle/working
!rm -rf TRELLIS.2
!git clone -b main --recursive https://github.com/microsoft/TRELLIS.2.git 2>&1 | tail -5
%cd /kaggle/working/TRELLIS.2

In [ ]:
import os, sys

os.environ['CUDA_HOME'] = '/usr/local/cuda'

# Remove Kaggle's pre-installed torch/torchvision to avoid conflicts
!pip uninstall -y torch torchvision torchaudio xformers 2>/dev/null || true
!pip install -U pip setuptools wheel --quiet

# PyTorch 2.9.1 + CUDA 12.8
!pip install torch==2.9.1 torchvision==0.22.0 --index-url https://download.pytorch.org/whl/cu128 --quiet

# xformers attention backend (pre-compiled, fast)
!pip install xformers --index-url https://download.pytorch.org/whl/cu128 --quiet

# Install rembg for background removal
!pip install rembg --quiet

print('✅ Base dependencies installed')

In [ ]:
# Download and install pre-compiled CUDA wheels from Aero-Ex
# Torch 2.9.1 + Python 3.12 + CUDA 12.4
WHEEL_BASE = 'https://raw.githubusercontent.com/Aero-Ex/ComfyUI-Trellis2-GGUF/main/wheels/Linux/Torch291'

wheels = [
    'cumesh-1.0-cp312-cp312-linux_x86_64.whl',
    'nvdiffrast-0.4.0-cp312-cp312-linux_x86_64.whl',
    'nvdiffrec_render-0.0.0-cp312-cp312-linux_x86_64.whl',
    'flex_gemm-0.0.1-cp312-cp312-linux_x86_64.whl',
    'o_voxel-0.0.1-cp312-cp312-linux_x86_64.whl',
]

for w in wheels:
    url = f'{WHEEL_BASE}/{w}'
    !wget -q --no-check-certificate "{url}" -O "{w}"
    !pip install "{w}" --quiet
    print(f'  ✅ {w}')

# Install TRELLIS.2 Python package
!pip install -e /kaggle/working/TRELLIS.2 --quiet

print('\n✅ All CUDA extensions installed (pre-compiled)')

---
## Step 3: Download DINOv3 (Public Mirror)

TRELLIS.2 needs `facebook/dinov3-vitl16-pretrain-lvd1689m` for the vision encoder.
Instead of using the gated Facebook repo, we download from the public **Aero-Ex/Dinov3** mirror — no token required.

In [ ]:
import os, requests
from pathlib import Path

CACHE = Path.home() / '.cache' / 'huggingface' / 'hub'
SNAPSHOT = 'mirror'
DINOV3_DIR = CACHE / f'models--facebook--dinov3-vitl16-pretrain-lvd1689m' / 'snapshots' / SNAPSHOT
DINOV3_DIR.mkdir(parents=True, exist_ok=True)

BASE = 'https://huggingface.co/Aero-Ex/Dinov3/resolve/main/facebook/dinov3-vitl16-pretrain-lvd1689m'
files = ['config.json', 'model.safetensors', 'preprocessor_config.json']

for f in files:
    dest = DINOV3_DIR / f
    if dest.exists() and dest.stat().st_size > 0:
        print(f'  ⏭️  {f} (cached)')
        continue
    url = f'{BASE}/{f}'
    print(f'  ⬇️  {f}...')
    r = requests.get(url, stream=True)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    written = 0
    with open(dest, 'wb') as fh:
        for chunk in r.iter_content(chunk_size=8192):
            fh.write(chunk)
            written += len(chunk)
            if total:
                pct = written / total * 100
                print(f'    {pct:.0f}% ({written/1024**3:.2f}GB)', end='\r')
    print(f'  ✅ {f} ({written/1024**3:.2f}GB)')

# Write refs/main so HF hub recognizes this snapshot
refs_dir = CACHE / f'models--facebook--dinov3-vitl16-pretrain-lvd1689m' / 'refs'
refs_dir.mkdir(parents=True, exist_ok=True)
(refs_dir / 'main').write_text(SNAPSHOT + '\n')

print(f'\n✅ DINOv3 ready in HF cache (from public mirror)')

---
## Step 4: Patch Gradio App for Kaggle

Sets default resolution to 512³ (for 16GB VRAM) and enables public sharing.

In [ ]:
app_path = Path('/kaggle/working/TRELLIS.2/app.py')
text = app_path.read_text()
text = text.replace('value="1024")', 'value="512")')
text = text.replace(
    'demo.launch(css=css, head=head)',
    'demo.launch(css=css, head=head, share=True, debug=True)'
)
app_path.write_text(text)
print('✅ app.py patched (resolution=512, share=True)')

---
## Step 5: Launch TRELLIS.2

First launch downloads ~15GB of model weights from Hugging Face (5-10 min).
Then the Gradio UI opens at a public `*.gradio.live` URL.

In [ ]:
print('🚀 Launching TRELLIS.2 Gradio UI...')
print('Downloading model weights (~15GB) on first launch...\n')

%cd /kaggle/working/TRELLIS.2
!ATTN_BACKEND=xformers SPARSE_ATTN_BACKEND=xformers \
 PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True,garbage_collection_threshold:0.6' \
 python app.py 2>&1